In [ ]:
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
import rasterio.features as features
import pandas as pd
import sys
lib_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
sys.path.append(str(lib_dir))
import Robyn_river_floods
import matplotlib.pyplot as plt

import seaborn as sns

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
print("Catchments:", len(catchments))
catchments.head()

In [ ]:
damage_reduction_max = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_max.tif"

In [ ]:
with rasterio.open(damage_reduction_max) as src:
    print("CRS:", src.crs)
    print("dtype:", src.dtypes[0])
    print("NoData:", src.nodata)
    arr_max = src.read(1, masked=True)  # respect NoData
    data = src.read(1, masked=True)
    crs, transform, shape = src.crs, src.transform, src.shape
    px = max(abs(transform.a), abs(transform.e))  # pixel size in meters


In [ ]:
# 2) Compute avoided_ead sums (min/max) by numeric catchment_uid

zs_max = Robyn_river_floods.zonal_sum_only(damage_reduction_max, catchments, id_col="catchment_uid") \
           .rename(columns={"sum": "avoided_ead_max"})

result = (catchments.merge(zs_max, on="catchment_uid"))

In [ ]:
# --- AREA PER CATCHMENT (km²) -----------------------------------------------
result = result.copy()
result["area_km2"] = result.geometry.area / 1e6  # assumes current CRS is metres

# Rebuild your table (sorted by avoided_ead_max) and show top 30 with area
tbl = (result.drop(columns="geometry")
             .sort_values("avoided_ead_max", ascending=False))

# Put key columns first (keep any others that exist)
first = ["catchment_uid", "area_km2", "avoided_ead_max", "avoided_ead_max_usd_mn"]
         
cols = [c for c in first if c in tbl.columns] + [c for c in tbl.columns if c not in first]

cols = [c for c in first if c in tbl.columns] + [c for c in tbl.columns if c not in first]
tbl = tbl[cols]

pd.options.display.float_format = "{:,.2f}".format
print("Columns:", print(tbl.columns))
print(tbl.head(30).to_string(index=False))

In [ ]:
# 3) Table view, sorted by avoided_ead_max
tbl = (result.drop(columns="geometry")
             .sort_values("avoided_ead_max", ascending=False))
print("Columns:", list(tbl.columns))
print(tbl.head(30).to_string(index=False))

In [ ]:
# === Convert J$ -> US$ and print a tidy table (USD millions) ===
cols = ["avoided_ead_max"]

# Make sure the columns exist
missing = [c for c in cols if c not in result.columns]
if missing:
    raise KeyError(f"Missing columns in `result`: {missing}")

for c in cols:
    result[f"{c}_usd"]     = result[c] / Robyn_river_floods.FX_JMD_PER_USD
    result[f"{c}_usd_mn"]  = result[c] / Robyn_river_floods.FX_JMD_PER_USD / 1e6
    result[f"{c}_usd_bil"] = result[c] / Robyn_river_floods.FX_JMD_PER_USD / 1e9

tbl_usd = (
    result[["catchment_uid"] + [f"{c}_usd_mn" for c in cols]]
      .rename(columns={
          "avoided_ead_min_usd_mn": "min_usd_mn",
          "avoided_ead_max_usd_mn": "max_usd_mn",
          "avoided_ead_mid_usd_mn": "mid_usd_mn",
      })
      .sort_values("max_usd_mn", ascending=False, kind="stable")
)

# Temporary display format (restore after printing)
_old = pd.options.display.float_format
pd.options.display.float_format = "{:,.1f}".format
try:
    print("Columns:", list(tbl_usd.columns))
    print(tbl_usd.head(30).to_string(index=False))
finally:
    pd.options.display.float_format = _old

# CSV export (all rows)
out_csv = output_dir / "catchment_avoided_ead_usd_millions.csv"
tbl_usd.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
summary = pd.DataFrame({
    # "avoided_ead_min": stats(result, "avoided_ead_min"),
    "avoided_ead_max": Robyn_river_floods.stats(result, "avoided_ead_max"),
}).T.reset_index().rename(columns={"index": "metric"})

print(summary.to_string(index=False))

In [ ]:
r_path = damage_reduction_max  # or _min; repeat for each raster you need
with rasterio.open(r_path) as src:
    crs = src.crs
    transform = src.transform
    shape = src.shape

    # 1) Reproject catchments to raster CRS
    cats = catchments.to_crs(crs)[["catchment_uid", "geometry"]].copy()
    cats["catchment_uid"] = pd.to_numeric(cats["catchment_uid"], errors="coerce").astype("Int64")
    cats = cats.dropna(subset=["catchment_uid"])

    # 2) Rasterize catchment IDs onto the SAME grid as the damage raster
    shapes = ((geom, int(uid)) for geom, uid in zip(cats.geometry, cats["catchment_uid"]))
    cat_id_raster = features.rasterize(
        shapes=shapes,
        out_shape=shape,
        transform=transform,
        fill=0,                   # background = 0 (no catchment)
        all_touched=True,         # include any pixel touched by polygon (no “gaps” at edges)
        dtype="int32"
    )

    # 3) Read raster data as masked array (nodata already masked)
    data = src.read(1, masked=True)

# 4) Sum by catchment label (exact match to raster total, no double count)
valid = ~data.mask
labels = cat_id_raster[valid]
values = data.data[valid].astype("float64")
# ignore background label 0
keep = labels > 0
labels = labels[keep]; values = values[keep]

sums = np.bincount(labels, weights=values)
uids = np.nonzero(sums)[0]
catchment_sums = pd.DataFrame({"catchment_uid": uids, "avoided_ead_max": sums[uids]})

# 5) Sanity: equality with national total (up to tiny rounding)
national_total = float(values.sum())
catchment_total = float(catchment_sums["avoided_ead_max"].sum())
print(f"National total:  {national_total:,.2f}")
print(f"Catchment total: {catchment_total:,.2f}")
print(f"Δ:               {catchment_total - national_total:,.6f}")

In [ ]:
# ==== Per-catchment sums for MAX raster (USD + USD mn) ========================
df_max = Robyn_river_floods.sum_by_catchment_usd(
    damage_reduction_max,
    catchments,
    label="avoided_ead_max",
)

catchment_usd = df_max.sort_values("catchment_uid").reset_index(drop=True)

# Consistency checks: catchment sum vs national print
s_usd    = catchment_usd["avoided_ead_max_usd"].sum()
s_usd_mn = catchment_usd["avoided_ead_max_usd_mn"].sum()
print(f"Sum over catchments (avoided_ead_max, USD):    US${s_usd:,.2f}")
print(f"Sum over catchments (avoided_ead_max, USD mn): {s_usd_mn:,.4f}")

print(
    catchment_usd.head().to_string(
        index=False,
        formatters={
            "avoided_ead_max_usd":    lambda x: f"{x:,.2f}",
            "avoided_ead_max_usd_mn": lambda x: f"{x:,.4f}",
        },
    )
)

In [ ]:
cats = catchments.to_crs(crs)[["catchment_uid","geometry"]].copy()
cats["catchment_uid"] = pd.to_numeric(cats["catchment_uid"], errors="coerce").astype("Int64")
cats = cats.dropna(subset=["catchment_uid"])

# Try a tiny outward buffer (half a pixel) to catch edge slivers
cats_buf = cats.copy()
cats_buf["geometry"] = cats_buf.geometry.buffer(px * 0.5)

lab = features.rasterize(
    ((geom, int(uid)) for geom, uid in zip(cats_buf.geometry, cats_buf["catchment_uid"])),
    out_shape=shape, transform=transform, fill=0, all_touched=True, dtype="int32"
)

valid = ~data.mask
coverage = (lab[valid] > 0).mean() * 100.0
unlabeled_value_sum_jmd = float(data.data[valid & (lab==0)].sum())
print(f"Coverage of valid raster by catchments: {coverage:.2f}%")
print(f"Sum of unlabeled valid raster cells (J$): {unlabeled_value_sum_jmd:,.2f}")
print(f"…which is US$ {unlabeled_value_sum_jmd/150:,.2f}")

In [ ]:
# --- Raster national total (assumed J$) → USD mn ------------------------------
raster_usd_mn = float(data.sum()) / Robyn_river_floods.FX / 1e6


# --- Parquet totals: buildings-only and all sectors ---------------------------
damage_future = pd.read_parquet(base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_future/damage__future.parquet")
damage_future["avoided_ead"] = (
    pd.to_numeric(damage_future["baseline__fluvial__ead"], errors="coerce")
  - pd.to_numeric(damage_future["future__fluvial__ead"],   errors="coerce")
)

df = damage_future.assign(sector=lambda d: d["asset_class"].map(Robyn_river_floods.sector_map))

buildings_usd_mn = df.query('sector == "buildings"')["avoided_ead"].sum() / Robyn_river_floods.FX / 1e6
all_usd_mn       = df["avoided_ead"].sum() / Robyn_river_floods.FX / 1e6

# --- Print & assert -----------------------------------------------------------
print(f"Raster (damage_reduction_max):      US$ {raster_usd_mn:,.2f} million")
# print(f"Parquet buildings only:             US$ {buildings_usd_mn:,.2f} million")
# print(f"Parquet ALL sectors:                US$ {all_usd_mn:,.2f} million")

# diff = raster_usd_mn - buildings_usd_mn
# pct  = (diff / buildings_usd_mn * 100) if buildings_usd_mn else np.nan
# print(f"Difference (raster - buildings):    {diff:,.2f} USD mn  ({pct:.2f}%)")

# # Guardrail: raster should match buildings within ~2%
# assert abs(pct) < 2.0, "Raster total differs from parquet buildings by >2% — check units/pipeline."

In [ ]:
benefit_col = "avoided_ead_max_usd_mn"  # use this USD millions column, or "avoided_ead_max" if you want J$
tbl = result  # your catchment table with geometry

df = (
    tbl[["catchment_uid", benefit_col]]
    .dropna()
    .query(f"{benefit_col} > 0")
    .copy()
)

# rank by benefit descending
df = df.sort_values(benefit_col, ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1
df["cum_benefit"] = df[benefit_col].cumsum()
df["cum_share"] = df["cum_benefit"] / df[benefit_col].sum()
df["rank_share"] = df["rank"] / len(df)

# cumulative curve
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["rank_share"], df["cum_share"], color="darkgreen", lw=2)
ax.set_xlabel("Share of catchments (ranked by avoided EAD)")
ax.set_ylabel(f"Share of total {benefit_col} (cumulative)")
ax.grid(True, linestyle=":", alpha=0.5)
plt.show()


In [ ]:

benefit_col = "avoided_ead_max_usd_mn"   # or "avoided_ead_max"
area_col    = "area_km2"
tbl = result  # your catchment table

df = (
    tbl[["catchment_uid", benefit_col, area_col]]
    .dropna()
    .query(f"{benefit_col} > 0")
    .copy()
    .sort_values(benefit_col, ascending=False)
    .reset_index(drop=True)
)
df["rank"] = df.index + 1

N = 30  # top N to show
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(df["rank"].iloc[:N], df[benefit_col].iloc[:N], width=0.9,
        color="steelblue", label="Avoided EAD")

# label bars with catchment_uid
for x, uid, val in zip(df["rank"].iloc[:N], df["catchment_uid"].iloc[:N], df[benefit_col].iloc[:N]):
    ax1.text(x, val, str(int(uid)), ha="center", va="bottom", fontsize=7, rotation=0)

ax1.set_xlabel("Catchments ranked by avoided EAD")
ax1.set_ylabel("Avoided EAD (USD millions)")
ax1.set_xlim(0.5, N + 0.5)
ax1.tick_params(axis='y', labelcolor="steelblue")

# overlay area as a line on secondary axis
ax2 = ax1.twinx()
ax2.plot(df["rank"].iloc[:N], df[area_col].iloc[:N],
         color="darkorange", marker="o", label="Catchment Area")
ax2.set_ylabel("Catchment area (km²)")
ax2.tick_params(axis='y', labelcolor="darkorange")

fig.tight_layout()
plt.show()


out_path = (
    output_dir / "figures/avoided_ead_by_catchment_area_max.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")
# plt.show()  # if you still want to display
out_path



In [ ]:
# restorable area per catchment (already numeric)
restorable = pd.read_csv(
    base_path / "dphil_paper_2/results/catchment_attributes/catchment_landuse_by_category_stats.csv",
    usecols=["catchment_uid", "reforestable_of_catchment_pct"],
)

df = (
    result[["catchment_uid", benefit_col]]   # uses existing avoided EAD column
    .merge(restorable, on="catchment_uid", how="inner")
    .dropna(subset=[benefit_col, "reforestable_of_catchment_pct"])
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="reforestable_of_catchment_pct",
    y=benefit_col,
    ax=ax,
)
ax.set_xlabel("Restorable area (%)")
ax.set_ylabel(f"Avoided EAD ({benefit_col})")
ax.set_title("Avoided EAD vs Restorable Area by Catchment")
ax.grid(True, linestyle=":", alpha=0.5)

out_path = base_path / "dphil_paper_2/results/bcr_mca_results/avoided_ead_vs_restorable_area_max.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
out_path


In [ ]:
benefit_col = benefit_col if "benefit_col" in globals() else "avoided_ead_max_usd_mn"
rest_col = "reforestable_of_catchment_pct"

# load restorable area
restorable = pd.read_csv(
    base_path / "dphil_paper_2/results/catchment_attributes/catchment_landuse_by_category_stats.csv",
    usecols=["catchment_uid", rest_col],
)

# build table with avoided EAD + restorable area
tbl = (
    result[["catchment_uid", benefit_col]]
    .merge(restorable, on="catchment_uid", how="inner")
    .dropna(subset=[benefit_col, rest_col])
    .sort_values(benefit_col, ascending=False)
    .reset_index(drop=True)
)
tbl["rank"] = tbl.index + 1

N = 30
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(tbl["rank"].iloc[:N], tbl[benefit_col].iloc[:N], width=0.9,
        color="steelblue", label="Avoided EAD")

# label bars with catchment_uid
for x, uid, val in zip(tbl["rank"].iloc[:N], tbl["catchment_uid"].iloc[:N], tbl[benefit_col].iloc[:N]):
    ax1.text(x, val, str(int(uid)), ha="center", va="bottom", fontsize=7)

ax1.set_xlabel("Catchments ranked by avoided EAD")
ax1.set_ylabel("Avoided EAD (USD millions)")
ax1.set_xlim(0.5, N + 0.5)
ax1.tick_params(axis='y', labelcolor="steelblue")

# overlay restorable area on secondary axis
ax2 = ax1.twinx()
ax2.plot(tbl["rank"].iloc[:N], tbl[rest_col].iloc[:N],
         color="darkorange", marker="o", label="Restorable area")
ax2.set_ylabel("Restorable area (%)")
ax2.tick_params(axis='y', labelcolor="darkorange")

fig.tight_layout()
out_path = output_dir / "figures/avoided_ead_by_restorable_area_max.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
out_path
